In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math


In [4]:
raw = np.load(r"/content/PEMS08.npz")["data"]     # (time, sensors, features)

data = raw[:,:,0]   # traffic flow only
print(data.shape)


(17856, 170)


In [5]:
scaler = MinMaxScaler()
data = scaler.fit_transform(data)


In [6]:
from torch.utils.data import Dataset

class TrafficDataset(Dataset):
    def __init__(self,data):
        self.data=data

    def __len__(self):
        return len(self.data)-SEQ_LEN-PRED_LEN

    def __getitem__(self,i):
        x = self.data[i:i+SEQ_LEN]
        y = self.data[i+SEQ_LEN:i+SEQ_LEN+PRED_LEN]
        return torch.tensor(x,dtype=torch.float32),torch.tensor(y,dtype=torch.float32)


In [7]:
SEQ_LEN = 72
PRED_LEN = 12
SENSORS = data.shape[1]
BATCH = 16
dataset = TrafficDataset(data)


In [8]:
split = int(0.8*len(dataset))

train_set = torch.utils.data.Subset(dataset, range(split))
val_set   = torch.utils.data.Subset(dataset, range(split,len(dataset)))

train_loader = DataLoader(train_set,batch_size=BATCH,shuffle=True)
val_loader   = DataLoader(val_set,batch_size=BATCH)


In [9]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)        # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x

In [10]:
class CNNLLM(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn2d = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU()
        )

        self.conv1d = nn.Conv1d(128*SENSORS,512,3,padding=1)

        self.project = nn.Linear(512,256)

        self.pos = PositionalEncoding(256)

        enc = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            dim_feedforward=1024,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(enc,num_layers=6)

        self.fc = nn.Linear(256,SENSORS*PRED_LEN)

    def forward(self,x):
        x = x.unsqueeze(1)
        x = self.cnn2d(x)

        B,C,T,S = x.shape
        x = x.permute(0,2,1,3).reshape(B,T,-1)

        x = self.conv1d(x.permute(0,2,1)).permute(0,2,1)

        x = self.project(x)
        x = self.pos(x)

        x = self.transformer(x)

        out = self.fc(x[:,-1])
        return out.reshape(B,PRED_LEN,S)


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNNLLM().to(device)

opt = torch.optim.Adam(model.parameters(),lr=3e-4)
loss_fn = nn.L1Loss()


In [12]:
patience = 8
best_val = 1e9
counter = 0

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt,mode='min',factor=0.5,patience=3
)


In [13]:
EPOCHS = 60

for ep in range(EPOCHS):

    model.train()
    train_loss = 0

    for x,y in train_loader:
        x,y = x.to(device),y.to(device)

        pred = model(x)
        loss = loss_fn(pred,y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x,y in val_loader:
            x,y = x.to(device),y.to(device)
            pred = model(x)
            val_loss += loss_fn(pred,y).item()

    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    print(ep,train_loss,val_loss)

    if val_loss < best_val:
        best_val = val_loss
        counter = 0
        torch.save(model.state_dict(),"best_model.pt")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break


0 0.08875419616833312 0.06284504357073874
1 0.05004725563345127 0.048517979330679764
2 0.04338493513695569 0.04441141761471872
3 0.040148644654669025 0.042632694017860386
4 0.0382267418197022 0.04217156629190851
5 0.03663992881565381 0.04048270095691018
6 0.035607439047825616 0.041057993887585374
7 0.034608749940957875 0.04013881677836848
8 0.03394259957657421 0.04080807970577826
9 0.033058242912166434 0.03970739554463481
10 0.0325601634279279 0.040180104680259135
11 0.032078951280836446 0.0396231010455989
12 0.03158230149722475 0.03950625271306712
13 0.031288010486232955 0.03970691858218657
14 0.030853866581690863 0.039744980892191556
15 0.03063973372946708 0.03909972100414236
16 0.03034782117385631 0.03911851530251482
17 0.030064642240782556 0.03969083098285402
18 0.029772044286232428 0.040084185237440825
19 0.029615620664489552 0.03982366148733237
20 0.028220876781070863 0.03899612630646459
21 0.028014973428255395 0.03937484643468007
22 0.027915787261871976 0.039157887444635144
23 0

In [14]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

preds,acts=[],[]

with torch.no_grad():
    for x,y in val_loader:
        x=x.to(device)
        p=model(x).cpu().numpy()
        preds.append(p)
        acts.append(y.numpy())

preds=np.concatenate(preds)
acts=np.concatenate(acts)

preds = scaler.inverse_transform(preds.reshape(-1,SENSORS)).reshape(preds.shape)
acts  = scaler.inverse_transform(acts.reshape(-1,SENSORS)).reshape(acts.shape)

mae = mean_absolute_error(acts.flatten(),preds.flatten())
rmse = np.sqrt(mean_squared_error(acts.flatten(),preds.flatten()))

print("MAE:",mae)
print("RMSE:",rmse)


MAE: 17.58196449279785
RMSE: 28.033489363240285
